## Notebook 05: Baseline Models

### Purpose
Establish benchmark accuracy that all ML models must beat.
A model that cannot beat a simple baseline is not useful.

### Baselines we build
1. Seasonal Naive — predict today = same day last week (lag_7)
2. Moving Average — predict today = rolling mean of last 28 days

### Why this matters
If LightGBM achieves 14% SMAPE vs 25% naive baseline,
that 11% gap is your headline result in interviews and resume.
Without a baseline, your model accuracy has no meaning.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/features/model_ready.parquet")
print(f"Loaded: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

Loaded: (27952032, 29)
Date range: 2013-01-30 00:00:00 to 2017-08-15 00:00:00


In [2]:
# SMAPE Function

# SMAPE = Symmetric Mean Absolute Percentage Error
# WHY we use SMAPE instead of MAPE:
# MAPE breaks when actual sales = 0 (division by zero)
# SMAPE handles zeros gracefully with the +1e-8 guard
# Formula: 100 * mean(2*|actual-pred| / (|actual|+|pred|+1e-8))

def smape(y_true, y_pred):
    """
    Symmetric Mean Absolute Percentage Error.
    Handles zero values. Range: 0% (perfect) to 200% (worst).
    """
    y_true = np.array(y_true, dtype=np.float32)
    y_pred = np.array(y_pred, dtype=np.float32)
    return 100 * np.mean(
        2 * np.abs(y_true - y_pred) /
        (np.abs(y_true) + np.abs(y_pred) + 1e-8)
    )

def mae(y_true, y_pred):
    return np.mean(np.abs(np.array(y_true) - np.array(y_pred)))

def bias(y_true, y_pred):
    """Positive = over-forecasting. Negative = under-forecasting."""
    return np.mean(np.array(y_pred) - np.array(y_true))

print("Metrics defined")
print("Testing SMAPE:")
print(f"  Perfect forecast:    {smape([10,20,30],[10,20,30]):.1f}%")
print(f"  50% off every time:  {smape([10,20,30],[15,30,45]):.1f}%")
print(f"  Zero actual handled: {smape([0,0,10],[1,1,10]):.1f}%")

Metrics defined
Testing SMAPE:
  Perfect forecast:    0.0%
  50% off every time:  40.0%
  Zero actual handled: 133.3%


In [3]:
# Define Evaluation Period

# WHY: we evaluate baselines on the last 28 days only
# This simulates real-world forecasting:
# train on historical data, evaluate on most recent period
# Same evaluation window will be used for all models
# so comparisons are fair

EVAL_DAYS = 28
cutoff_date = df["date"].max() - pd.Timedelta(days=EVAL_DAYS)

train = df[df["date"] <= cutoff_date].copy()
test  = df[df["date"] >  cutoff_date].copy()

print(f"Cutoff date:    {cutoff_date.date()}")
print(f"Train rows:     {len(train):,}")
print(f"Test rows:      {len(test):,}")
print(f"Test date range: {test['date'].min().date()} "
      f"to {test['date'].max().date()}")
print(f"Unique test dates: {test['date'].nunique()}")

Cutoff date:    2017-07-18
Train rows:     27,351,942
Test rows:      600,090
Test date range: 2017-07-19 to 2017-08-15
Unique test dates: 28


In [4]:
# Baseline 1: Seasonal Naive

# SEASONAL NAIVE: predict today = same day last week
# WHY this is a strong baseline for retail:
# Weekly patterns are very consistent in grocery data
# Our EDA decomposition confirmed strong weekly seasonality
# If a model can't beat "just use last week's sales",
# it's not useful

# lag_7 is already in our feature store — use it directly
seasonal_naive_preds = test["lag_7"].values
actual = test["unit_sales"].values

naive_smape = smape(actual, seasonal_naive_preds)
naive_mae   = mae(actual, seasonal_naive_preds)
naive_bias  = bias(actual, seasonal_naive_preds)

print("=" * 40)
print("BASELINE 1: Seasonal Naive (lag_7)")
print("=" * 40)
print(f"SMAPE: {naive_smape:.2f}%")
print(f"MAE:   {naive_mae:.4f}")
print(f"Bias:  {naive_bias:.4f}")
print()
print("INTERPRETATION:")
print(f"On average, predicting last week's sales")
print(f"gives {naive_smape:.1f}% error.")
print("All our ML models must beat this number.")

BASELINE 1: Seasonal Naive (lag_7)
SMAPE: 81.94%
MAE:   4.1821
Bias:  -0.3690

INTERPRETATION:
On average, predicting last week's sales
gives 81.9% error.
All our ML models must beat this number.


In [5]:
# Baseline 2: Moving Average

# MOVING AVERAGE: predict today = average of last 28 days
# rolling_mean_28 already computed in feature store

ma_preds = test["rolling_mean_28"].values

ma_smape = smape(actual, ma_preds)
ma_mae   = mae(actual, ma_preds)
ma_bias  = bias(actual, ma_preds)

print("=" * 40)
print("BASELINE 2: Moving Average (28-day)")
print("=" * 40)
print(f"SMAPE: {ma_smape:.2f}%")
print(f"MAE:   {ma_mae:.4f}")
print(f"Bias:  {ma_bias:.4f}")

BASELINE 2: Moving Average (28-day)
SMAPE: 49.99%
MAE:   3.4136
Bias:  0.2714


In [7]:
# Baseline Comparison Table

import json
from datetime import datetime

# Summary table
results = {
    "Seasonal Naive": {
        "smape": round(naive_smape, 2),
        "mae":   round(naive_mae, 4),
        "bias":  round(naive_bias, 4)
    },
    "Moving Average 28d": {
        "smape": round(ma_smape, 2),
        "mae":   round(ma_mae, 4),
        "bias":  round(ma_bias, 4)
    }
}

print("=" * 50)
print("BASELINE COMPARISON")
print("=" * 50)
print(f"{'Model':<25} {'SMAPE':>8} {'MAE':>10} {'Bias':>10}")
print("-" * 50)
for model, metrics in results.items():
    print(f"{model:<25} "
          f"{metrics['smape']:>7.2f}% "
          f"{metrics['mae']:>10.4f} "
          f"{metrics['bias']:>10.4f}")
print("=" * 50)
print(f"\nTarget: LightGBM must beat "
      f"{min(naive_smape, ma_smape):.2f}% SMAPE")

# Save baseline results
# WHY we cast to float(): json.dump doesn't understand numpy float32
# Must convert to Python native float first

baseline_log = {
    "computed_date": datetime.now().isoformat(),
    "eval_period_days": int(EVAL_DAYS),
    "cutoff_date": str(cutoff_date.date()),
    "baselines": {
        model: {k: float(v) for k, v in metrics.items()}
        for model, metrics in results.items()
    }
}

with open("../outputs/baselines.json", "w") as f:
    json.dump(baseline_log, f, indent=2)

print("\nSaved to outputs/baselines.json")

BASELINE COMPARISON
Model                        SMAPE        MAE       Bias
--------------------------------------------------
Seasonal Naive              81.94%     4.1821    -0.3690
Moving Average 28d          49.99%     3.4136     0.2714

Target: LightGBM must beat 49.99% SMAPE

Saved to outputs/baselines.json


In [8]:
# Breakdown by Family

# WHY: overall SMAPE hides where the model struggles
# GROCERY I (29.5% of volume) might have 12% SMAPE
# while a small family has 60% SMAPE
# Knowing this tells you where your model needs work

family_results = []

for family in df["family"].cat.categories:
    mask = test["family"] == family
    if mask.sum() < 10:
        continue
    f_actual = test.loc[mask, "unit_sales"].values
    f_pred   = test.loc[mask, "lag_7"].values
    family_results.append({
        "family":  family,
        "smape":   round(smape(f_actual, f_pred), 2),
        "rows":    int(mask.sum())
    })

family_df = (
    pd.DataFrame(family_results)
    .sort_values("smape")
    .reset_index(drop=True)
)

print("Seasonal Naive SMAPE by Product Family:")
print(family_df.to_string(index=False))
print(f"\nBest family:  {family_df.iloc[0]['family']} "
      f"({family_df.iloc[0]['smape']}% SMAPE)")
print(f"Worst family: {family_df.iloc[-1]['family']} "
      f"({family_df.iloc[-1]['smape']}% SMAPE)")

Seasonal Naive SMAPE by Product Family:
                    family  smape   rows
                      EGGS  62.27   7068
                   PRODUCE  62.51  51432
            PREPARED FOODS  65.45   2165
                     MEATS  65.97  10987
                   POULTRY  66.27   9437
                   SEAFOOD  69.04   1391
              BREAD/BAKERY  73.20  20874
                     DAIRY  73.91  45008
                      DELI  79.39  15845
              FROZEN FOODS  79.73   6614
                 HOME CARE  80.88  17440
                 GROCERY I  81.98 202861
                 BEVERAGES  87.45  94432
                  CLEANING  87.96  65627
             PERSONAL CARE  91.07  20486
                GROCERY II  97.78   1544
          LIQUOR,WINE,BEER  99.08   5178
                    BEAUTY 105.85    910
                 MAGAZINES 106.65    905
           LAWN AND GARDEN 109.58   2029
   PLAYERS AND ELECTRONICS 112.92   1819
               CELEBRATION 115.99   2407
                L

# ── NOTEBOOK SUMMARY ──────────────────────────────────────────────
# 1. What I did:
#    Computed two baselines on last 28 days of data.
#
# 2. What I found:
#    Seasonal Naive SMAPE = 81.94% (high due to missing date gaps)
#    Moving Average SMAPE = 49.99% (stronger baseline)
#    Best family:  EGGS (62.27%)
#    Worst family: BABY CARE (163.27% — only 49 rows, unreliable)
#    GROCERY I (highest volume): 81.98%
#
# 3. Decision made:
#    Moving Average (49.99%) is the real benchmark.
#    LightGBM target: beat 49.99% SMAPE overall,
#    and specifically improve GROCERY I below 81.98%.
#
# 4. What I would do with more time:
#    Compute volume-weighted SMAPE so GROCERY I counts more
#    than BABY CARE in the overall score.
#
# 5. Question for a senior DS:
#    Should we exclude families with < 100 test rows
#    from the evaluation to get a more reliable benchmark?
print("Summary written")